# 🏆 LifeMate - Zalo AI Challenge 2026 (Kaggle 2x Tesla T4 GPUs)

Notebook này được thiết kế riêng để chạy pipeline Agentic Voice Assistant trên **Kaggle Notebooks với 2x GPU Tesla T4 (Tổng cộng ~30GB VRAM)**.

---

### ⚡ Điểm nổi bật khi chạy trên Kaggle 2x T4:
- **Tổng 30GB VRAM (2 x 15GB)**: Model `Qwen3.5-4B` (hoặc `Qwen2.5-3B`) được chia tải tự động qua 2 GPU bằng `device_map="auto"` (mỗi GPU chỉ chiếm ~4.5GB VRAM).
- **Không lo OOM**: Dư hơn **10GB VRAM trên mỗi GPU** cho activation và prompt dài.
- **Chạy ngầm (Commit & Run)**: Bạn có thể chọn **Save Version -> Save & Run All (Commit)** rồi tắt máy, Kaggle sẽ tự chạy nền trên đám mây và xuất file `submission.jsonl` vào mục Output để tải về nộp bài.

---

### ⚙️ Thiết lập quan trọng trên Kaggle trước khi chạy:
Ở thanh menu bên phải (**Session options**):
1. **ACCELERATOR**: Chọn **`GPU T4 x2`**
2. **INTERNET**: Bật **`Internet On`** (Bắt buộc bật để clone Git và tải weights mô hình)
3. **INPUT**: Nhấn **`+ Add Data`** để thêm dataset `public_test` mà bạn đã upload lên Kaggle Datasets.

## Bước 1: Kiểm tra 2x GPU Tesla T4

In [ ]:
# Kiểm tra 2 GPU Tesla T4 (GPU 0 và GPU 1)
!nvidia-smi

## Bước 2: Clone Codebase từ Git Repo

In [ ]:
import os

# Di chuyển vào thư mục làm việc ghi được (/kaggle/working)
%cd /kaggle/working
!rm -rf LifeMate

# Thay <URL_REPO_CỦA_BẠN> bằng URL git của bạn (ví dụ: https://github.com/username/LifeMate.git)
# Nếu repo private, bạn có thể truyền token: https://<TOKEN>@github.com/username/LifeMate.git
REPO_URL = "https://github.com/your-username/LifeMate.git"  # <-- Cập nhật URL repo của bạn
BRANCH = "colab-t4"                                         # <-- Branch chứa mã nguồn mới nhất

!git clone -b $BRANCH $REPO_URL LifeMate
%cd /kaggle/working/LifeMate

## Bước 3: Cài đặt thư viện phụ thuộc

In [ ]:
# Cài đặt các package cần thiết: transformers, accelerate, bitsandbytes, qwen-agent, sherpa-onnx
!pip install -q -r requirements.txt

## Bước 4: Tự động tìm đường dẫn Dataset trong `/kaggle/input`

In [ ]:
from pathlib import Path

# Quét thư mục chứa public_test trong /kaggle/input
kaggle_input = Path("/kaggle/input")
found_dirs = list(kaggle_input.rglob("public_test.jsonl"))

if found_dirs:
    data_dir = str(found_dirs[0].parent)
    print(f"✅ Đã tìm thấy thư mục dữ liệu public_test: {data_dir}")
else:
    # Fallback thủ công nếu bạn đặt tên dataset khác
    data_dir = "/kaggle/input/zaic2026-public-test/public_test"  # <-- Sửa đường dẫn nếu cần
    print(f"⚠️ Không tự động tìm thấy, sử dụng đường dẫn mặc định: {data_dir}")

os.environ["DATA_DIR"] = data_dir

## Bước 5: Chạy thử nghiệm 2 mẫu đầu tiên (Sanity Check)
Chạy thử 2 mẫu để xác nhận mô hình phân bổ đều trên 2 GPU và inference trơn tru.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python main.py \
    --mode public_test \
    --num-samples 2 \
    --data-dir "$DATA_DIR" \
    --output /kaggle/working/test_submission.jsonl

## Bước 6: Chạy toàn bộ 634 mẫu public_test & Xuất submission.jsonl
- Kết quả được lưu trực tiếp tại `/kaggle/working/submission.jsonl`.
- Với 2x T4 GPU, quá trình chạy 634 mẫu chỉ mất khoảng **25 - 40 phút**!

In [ ]:
# Chạy toàn bộ 634 mẫu
!python main.py \
    --mode public_test \
    --num-samples -1 \
    --data-dir "$DATA_DIR" \
    --output /kaggle/working/submission.jsonl

## Bước 7: Kiểm tra kết quả hoàn thành
Sau khi chạy xong, file `submission.jsonl` sẽ nằm tại `/kaggle/working/submission.jsonl`. Bạn có thể tải trực tiếp từ tab **Output** bên phải màn hình Kaggle.

In [ ]:
# Kiểm tra số dòng (chuẩn 634 mẫu)
!wc -l /kaggle/working/submission.jsonl

# Xem thử 3 dòng đầu
!head -n 3 /kaggle/working/submission.jsonl

print("🎉 Hoàn thành xuất file nộp bài submission.jsonl trên Kaggle 2x T4!")